In [ ]:
from typing import TypedDict, Optional
import os
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import tool
import requests
from IPython.display import Image, display
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, START, END, add_messages
from typing import TypedDict, Annotated

In [ ]:
class GraphStateData(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

llm = ChatOpenAI(
    model="gpt-4.1-mini",  # Your Azure deployment name
    base_url="https://openai-rg-nadeem.openai.azure.com/openai/v1",
    #add your api key below
    # api_key=
    )
search_tool = DuckDuckGoSearchRun()

In [6]:
@tool
def get_weather_tool(city: str) -> str:
    """This tool returns weather information for a given city."""
    if "delhi" in city.lower():
        return "Delhi is very hot 52C"
    return f"{city} is very cold 32 degree celsius"

# Example usage
print(type(get_weather_tool))
print(get_weather_tool.invoke({"city": "what is the weather in delhi"}))

@tool
def get_pokemon_info(pokemon: str) -> str:
    """This tool fetches information about a Pokemon from the PokeAPI."""
    url = f"https://pokeapi.co/api/v2/pokemon/{pokemon.lower()}"
    try:
        response = requests.get(url, timeout=10)
    except requests.RequestException as err:
        return f"Request error: {err}"

    if response.status_code == 200:
        data = response.json()
        name = data["name"]
        height = data["height"]
        weight = data["weight"]
        types = [t["type"]["name"] for t in data["types"]]
        return f"Name: {name}, Height: {height}, Weight: {weight}, Types: {', '.join(types)}"
    return f"Error: Pokemon '{pokemon}' not found."

# Example usage
print(type(get_pokemon_info))
print(get_pokemon_info.invoke({"pokemon": "ditto"}))

<class 'langchain_core.tools.structured.StructuredTool'>
Delhi is very hot 52C
<class 'langchain_core.tools.structured.StructuredTool'>
Name: ditto, Height: 3, Weight: 40, Types: normal


In [7]:
tools = [get_weather_tool, get_pokemon_info]
llm_with_tools = llm.bind_tools(tools)

In [8]:
def chatbot(state: GraphStateData) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {
        "messages": [response]
    }

tool_node = ToolNode(tools=tools)

In [9]:
workflow = StateGraph(GraphStateData)

workflow.add_edge(START, "chatbot")
workflow.add_node("chatbot", chatbot)

workflow.add_node("tools", tool_node)
workflow.add_conditional_edges("chatbot", tools_condition)
workflow.add_edge("tools", "chatbot")

graph = workflow.compile()

result = graph.invoke({
    "messages": [HumanMessage(content="Tell the weather of delhi")]
})

result["messages"][-1].content

'The weather in Delhi is very hot with a temperature of 52°C.'